# Automate Data Extraction & Finalize Raw Dataset

**Objective:**
The objective of this phase is to automate the data collection process. Instead of manually extracting military statistics for over 140 countries, we utilize a Python script to systematically navigate the Global Firepower index. The script reads a list of country URLs, extracts key metrics such as manpower, defense budgets, and equipment counts, and compiles this information into an initial raw dataset.




In [ ]:
import pandas as pd
import requests
import re
import time
from bs4 import BeautifulSoup

In [ ]:
# 1. Get Country List
countries_url = "https://www.globalfirepower.com/countries-listing.php"
headers = {"User-Agent": "Mozilla/5.0"}
response = requests.get(countries_url, headers=headers, timeout=10)
response.raise_for_status()
soup = BeautifulSoup(response.text, "html.parser")

country_tags = soup.find_all("a", href=True)
ids = []

for a in country_tags:
    href = str(a["href"])
    if "country-military-strength-detail.php?country_id=" in href:
        cid = href.split("country_id=")[1]
        ids.append(cid)

ids = list(set(ids))

# 2. Key Metric Mapping
key_map = {
    "total_population": "Total Population:",
    "gdp": "Purchasing Power Parity:",
    "defense_budget": "Defense Budget:",
    "total_manpower": "Available Manpower",
    "active_personnel": "Active Personnel",
    "reserve_personnel": "Reserve Personnel",
    "air_force_personnel": "Air Force Personnel*",
    "army_personnel": "Army Personnel*",
    "navy_personnel": "Navy Personnel*",
    "total_aircraft": "Aircraft Total:",
    "attack_aircraft": "Attack Types:",
    "fighter_aircraft": "Fighters:",
    "transport_aircraft": "Transports (Fixed-Wing):",
    "helicopters": "Helicopters:",
    "special_mission_aircraft": "Special-Mission:",
    "trainer_aircraft": "Trainers:",
    "attack_helicopters": "Attack Helicopters:",
    "tanker_aircraft": "Tanker Fleet:",
    "naval_assets": "Total Assets:",
    "aircraft_carriers": "Aircraft Carriers:",
    "helicopter_carriers": "Helicopter Carriers:",
    "destroyers": "Destroyers:",
    "frigates": "Frigates:",
    "corvettes": "Corvettes:",
    "submarines": "Submarines:",
    "patrol_vessel": "Patrol Vessels:",
    "mine_warfare": "Mine Warfare:",
    "tanks": "Tanks:",
    "self_propelled_artillery": "Self-Propelled Artillery:",
    "towed_artillery": "Towed Artillery:",
    "rocket_artillery": "MLRS (Rocket Artillery):",
    "external_debt": "External Debt:"
}

# 3. Automation Function
def automation(ids, key_map):
    base_url = "https://www.globalfirepower.com/country-military-strength-detail.php?country_id="
    all_countries = {}

    for cid in ids:
        try:
            url = base_url + cid
            response = requests.get(url, headers=headers, timeout=10)
            response.raise_for_status()
            soup = BeautifulSoup(response.text, "html.parser")

            country_name = cid.replace("-", " ").title()

            # Extract Power Index
            power_index = None
            info_block = soup.select_one("span.textNormal.textDkGray")
            if info_block:
                match = re.search(r"\d+\.\d+", info_block.text)
                if match:
                    power_index = match.group()

            # Extract Specifications
            specs_containers = soup.find_all("div", class_="specsGenContainers")
            data_found = {}

            for container in specs_containers:
                tag = container.select_one("span.textLarge.textBold.textShadow")
                value_tags = container.select("span.textWhite.textShadow")

                if tag and value_tags:
                    name = tag.text.strip()
                    values = [v.text.strip() for v in value_tags]
                    data_found[name] = values[-1]

            # Map the results
            result = {}
            for new_key, old_key in key_map.items():
                if old_key in data_found:
                    result[new_key] = data_found[old_key]
                else:
                    result[new_key] = None

            result["power_index"] = power_index
            all_countries[country_name] = result

            print(f"Scraped: {country_name}")
            time.sleep(1)

        except Exception as e:
            print(f"Error scraping {cid}: {e}")

    return all_countries

# 4. Execution
all_countries_data = automation(ids, key_map)

df = pd.DataFrame.from_dict(all_countries_data, orient="index")
df.reset_index(inplace=True)
df.rename(columns={"index": "country"}, inplace=True)

df.to_csv("military_raw_data.csv", index=False)
print("Done. File saved as military_raw_data.csv")

**Takeaway:** Successfully extracted raw text metrics for 145 countries and exported the unstructured dataset to `military_raw_data.csv`.

# Data Cleaning & Standardization

**Objective:**
The objective of this phase is to prepare the raw data for mathematical analysis. Computers cannot perform calculations on text strings containing symbols (e.g., "$500" or "50%"). This step focuses on cleaning the dataset by removing formatting characters, standardizing column names for consistency, and handling missing information so that the dataset consists entirely of usable numeric values.

In [ ]:
import pandas as pd
import numpy as np
import sys
import re

In [ ]:
# Force UTF-8 output
try:
    sys.stdout.reconfigure(encoding='utf-8')
except AttributeError:
    pass

# --- CONFIGURATION ---
INPUT_FILE = "military_raw_data.csv"
OUTPUT_FILE = "military_cleaned.csv"

# --- STEP 1: LOAD RAW DATA ---
print(f"Step 1: Loading raw data from '{INPUT_FILE}'...")
try:
    df = pd.read_csv(INPUT_FILE)
    print("   -> Raw data loaded successfully.")
    print(f"   -> Initial Shape: {df.shape}")
except FileNotFoundError:
    raise Exception(f"ERROR: Could not find '{INPUT_FILE}'.")

# --- STEP 4: STANDARDIZE COLUMN NAMES ---
print("\nStep 4: Standardizing column names (snake_case)...")
df.columns = df.columns.str.lower().str.strip().str.replace(' ', '_')
print(f"   -> Columns standardized: {list(df.columns[:5])}...")

# --- STEP 2 & 3: CLEAN NUMERIC COLUMNS ---
print("\nStep 2 & 3: Cleaning numeric columns...")

def clean_numeric_value(x):
    if pd.isna(x): return np.nan
    x = str(x).strip().replace(',', '').replace('$', '').replace('%', '').replace('+', '')
    if x.lower() in ['n/a', 'nan', 'null', '', 'none']: return np.nan
    match = re.search(r'([\d.]+)', x)
    if match:
        return float(match.group(1))
    return np.nan

cols_to_clean = [col for col in df.columns if 'country' not in col]
for col in cols_to_clean:
    df[col] = df[col].apply(clean_numeric_value)

print(f"   -> Cleaned {len(cols_to_clean)} numeric columns.")

# --- STEP 5: HANDLING MISSING VALUES ---
print("\nStep 5: Handling missing values...")
null_counts = df.isnull().sum()
missing_cols = null_counts[null_counts > 0]
if not missing_cols.empty:
    print(f"   -> Found missing values in:\n{missing_cols}")
else:
    print("   -> No missing values found initially.")

df = df.fillna(0)
print("   -> Applied Rule: Filled missing values with 0.")

# --- STEP 6: VALIDATE CLEANED DATA ---
print("\nStep 6: Validating cleaned data...")

row_count = len(df)
if row_count < 140:
    print(f"   WARNING: Row count is {row_count}. Check scraping completeness.")
else:
    print(f"   [OK] Row count check passed: {row_count} countries.")

non_numeric_cols = df.select_dtypes(exclude=[np.number]).columns.tolist()
if len(non_numeric_cols) > 1:
    print(f"   WARNING: Unexpected non-numeric columns: {non_numeric_cols}")
else:
    print("   [OK] Data type check passed (Metrics are numeric).")

if df.isnull().sum().sum() == 0:
    print("   [OK] Null value check passed (0% missing).")
else:
    print("   WARNING: There are still null values remaining.")

# --- STEP 7: EXPORT ---
print(f"\nStep 7: Exporting to '{OUTPUT_FILE}'...")
df.to_csv(OUTPUT_FILE, index=False)
print(f"[SUCCESS] Cleaned dataset saved as '{OUTPUT_FILE}'.")
print("   -> Ready for KPI Engineering (Task 7).")

**Takeaway:** Cleaned 30 numeric columns, specifically identifying and zero-imputing exactly 32 missing values (22%) across all Naval Asset fields due to landlocked countries.